In [ ]:
import numpy as np
import os
import cv2
import time
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import urllib.request # Added for in-memory image download

# define some parameters
CONFIDENCE = 0.5
font_scale = 1
thickness = 1

# loading the YOLOv8 model with the default weight file
model = YOLO("yolov8n.pt")

# Download the coco.names file if it doesn't exist
if not os.path.exists("/content/coco.names"):
    !wget -q https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names -O /content/coco.names

# loading all the class labels (objects)
labels = open("/content/coco.names").read().strip().split("\n")

# generating colors for each object for later plotting
colors = np.random.randint(0, 255, size=(len(labels), 3), dtype="uint8")

path_name = "/content/bus.jpg" # Updated to bus.jpg

# Download image data directly and decode using cv2
try:
    with urllib.request.urlopen("https://ultralytics.com/images/bus.jpg") as url: # Updated URL
        image_data = url.read()
    image_np = np.frombuffer(image_data, np.uint8)
    image = cv2.imdecode(image_np, cv2.IMREAD_COLOR)
except Exception as e:
    raise FileNotFoundError(f"Error: Could not download or decode image from URL. Details: {e}")

# Check if image was loaded successfully (imdecode can also return None on failure)
if image is None:
    raise FileNotFoundError(f"Error: cv2.imdecode failed to load image from URL. The image might be corrupted or in an unsupported format. The model is currently inferring on its default assets (bus.jpg, zidane.jpg) due to this loading failure.")

file_name = os.path.basename(path_name) # "bus.jpg"
filename, ext = file_name.split(".") # "bus", "jpg"

# measure how much it took in seconds
start = time.perf_counter()

# run inference on the image
# see: https://docs.ultralytics.com/modes/predict/#arguments for full list of arguments
results = model.predict(image, conf=CONFIDENCE)[0]
time_took = time.perf_counter() - start
print(f"Time took: {time_took:.2f}s")
print(results.boxes.data)

# loop over the detections
for data in results.boxes.data.tolist():
    # get the bounding box coordinates, confidence, and class id
    xmin = int(data[0])
    ymin = int(data[1])
    xmax = int(data[2])
    ymax = int(data[3])
    confidence = data[4]
    class_id = int(data[5])

    # draw a bounding box rectangle and label on the image
    color = [int(c) for c in colors[class_id]]
    cv2.rectangle(image, (xmin, ymin), (xmax, ymax), color=color, thickness=thickness)
    text = f"{labels[class_id]}: {confidence:.2f}"

    # calculate text width & height to draw the transparent boxes as background of the text
    (text_width, text_height) = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fontScale=font_scale, thickness=thickness)[0]
    text_offset_x = xmin
    text_offset_y = ymin - 5

    box_coords = ((text_offset_x, text_offset_y), (text_offset_x + text_width + 2, text_offset_y - text_height))
    overlay = image.copy()
    cv2.rectangle(overlay, box_coords[0], box_coords[1], color=color, thickness=cv2.FILLED)
    # add opacity (transparency to the box)
    image = cv2.addWeighted(overlay, 0.6, image, 0.4, 0)
    # now put the text (label: confidence %)
    cv2.putText(image, text, (xmin, ymin - 5), cv2.FONT_HERSHEY_SIMPLEX, fontScale=font_scale, color=(0, 0, 0), thickness=thickness)

# display output image
cv2_imshow(image)
cv2.waitKey(0)

# save output image to disk
cv2.imwrite(filename + "_yolo8." + ext, image)


0: 640x480 3 persons, 1 bus, 169.9ms
Speed: 6.1ms preprocess, 169.9ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 480)
Time took: 0.22s
tensor([[2.2871e+01, 2.3128e+02, 8.0500e+02, 7.5684e+02, 8.7345e-01, 5.0000e+00],
        [4.8550e+01, 3.9855e+02, 2.4535e+02, 9.0270e+02, 8.6569e-01, 0.0000e+00],
        [6.6947e+02, 3.9219e+02, 8.0972e+02, 8.7704e+02, 8.5284e-01, 0.0000e+00],
        [2.2152e+02, 4.0580e+02, 3.4497e+02, 8.5754e+02, 8.2522e-01, 0.0000e+00]])
